# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 1 1.3333333333333333 — isna().sum() считает None 1 пропуск,тт а fillna(0).mean() подставляет 0 и делит (1 + 0 + 3) / 3 = 1.333
2. 2 - " a".strip().lower() и "A ".strip().lower() дают "а", а "b" остаётся "b" - итого 2 уникальных значения
3. 3 - df.drop_duplicates() без inplace=True или присваивания не меняет df, поэтому len(df) остаётся 3
4. 1005 убираем пробелы, приводим к числам (1000 и 5) и суммируем: 1005
5. [9, 10] - после pd.to_datetime свойство .dt.month возвращает номера месяцев (9 и 10)

In [17]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [39]:
# 1
def clean(path):
    d = pd.read_csv(path)
    print("Старт:", len(d))
    d["amount"] = pd.to_numeric(d["amount"].str.replace(" ", ""))
    print("После amount:", len(d))

    CITY = {
        "алматы": "Алматы", "almaty": "Алматы",
        "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент",
        "караганда": "Караганда",
        "актобе": "Актобе",
        "атырау": "Атырау",
    }
    d["city"] = d["city"].str.strip().str.lower().map(CITY)
    print("После city:", len(d))
    d = d.drop_duplicates().copy()
    print("После drop_duplicates:", len(d))

    d["order_date"] = pd.to_datetime(d["order_date"])
    d["month"] = d["order_date"].dt.month
    print("После дат:", len(d))
    d = d[d["customer_id"] != "TEST"].copy()
    print("После удаления TEST:", len(d))

    print("NaN в amount:", d["amount"].isna().sum(),
          "| NaN в city:", d["city"].isna().sum(),
          "| order_id уникален:", d["order_id"].is_unique)
    return d
clean_df = clean("orders.csv")
print("Итого строк:", len(clean_df))

Старт: 5463
После amount: 5463
После city: 5463
После drop_duplicates: 5283
После дат: 5283
После удаления TEST: 5280
NaN в amount: 0 | NaN в city: 0 | order_id уникален: True
Итого строк: 5280


Для менеджера: После очистки остаётся 5 280 строк (убрано 180 дублей и 3 тестовые строки TEST), все суммы переведены в числа, города сведены к 6 названиям, а order_id уникален, на такой таблице уже можно корректно считать выручку и динамику

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [91]:
# 2
paid_c = clean_df[clean_df["status"] == "оплачен"]
rev_sep = paid_c.loc[paid_c["month"] == 9, "amount"].sum()
rev_oct = paid_c.loc[paid_c["month"] == 10, "amount"].sum()
pct_all = (rev_oct / rev_sep - 1) * 100
print(f"Всего оплаченных: сентябрь = {rev_sep:,.0f}, октябрь = {rev_oct:,.0f}, изменение = {pct_all:.1f}%")

for c in ["Алматы", "Астана", "Шымкент", "Караганда", "Актобе", "Атырау"]:
    sub = paid_c[paid_c["city"] == c]
    s_val = sub.loc[sub["month"] == 9, "amount"].sum()
    o_val = sub.loc[sub["month"] == 10, "amount"].sum()
    ch = (o_val / s_val - 1) * 100
    print(f"  {c}: сентябрь = {s_val:,.0f}, октябрь = {o_val:,.0f}, изменение = {ch:+.1f}%")

Всего оплаченных: сентябрь = 35,243,410, октябрь = 34,419,520, изменение = -2.3%
  Алматы: сентябрь = 14,501,070, октябрь = 14,643,020, изменение = +1.0%
  Астана: сентябрь = 9,034,220, октябрь = 8,850,530, изменение = -2.0%
  Шымкент: сентябрь = 4,558,650, октябрь = 4,044,010, изменение = -11.3%
  Караганда: сентябрь = 3,648,520, октябрь = 3,371,680, изменение = -7.6%
  Актобе: сентябрь = 2,576,960, октябрь = 2,540,420, изменение = -1.4%
  Атырау: сентябрь = 923,990, октябрь = 969,860, изменение = +5.0%


Для менеджера: По чистым данным выручка оплаченных заказов снизилась с 35 243 410  в сентябре до 34 419 520  в октябре (-2.3%, совпадает с уроком 4). Сильнее всех падает Шымкент (-11.3%, с 4 558 650  до 4 044 010 ), далее Караганда (-7.6%), тогда как Алматы (+1.0%) и Атырау (+5.0%) подросли, фокус внимания нужен на Шымкенте

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [49]:
# 3
df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
sep = df["order_date"] < "2026-10"
octo = ~sep
paid = df["status"] == "оплачен"
s0, o0 = df.loc[paid & sep, "amount"].sum(), df.loc[paid & octo, "amount"].sum()
print(f"Как считал менеджер: {len(df)} строк | окт/сент: {o0 / s0 - 1:.1%}")

df1 = df.drop_duplicates().copy()
sep1, octo1, paid1 = df1["order_date"] < "2026-10", ~(df1["order_date"] < "2026-10"), df1["status"] == "оплачен"
s1, o1 = df1.loc[paid1 & sep1, "amount"].sum(), df1.loc[paid1 & octo1, "amount"].sum()
print(f"Без дублей (убрано {len(df) - len(df1)}): {len(df1)} строк | окт/сент: {o1 / s1 - 1:.1%}")

df2 = df1[df1["customer_id"] != "TEST"].copy()
sep2, octo2, paid2 = df2["order_date"] < "2026-10", ~(df2["order_date"] < "2026-10"), df2["status"] == "оплачен"
s2, o2 = df2.loc[paid2 & sep2, "amount"].sum(), df2.loc[paid2 & octo2, "amount"].sum()
print(f"Без TEST (убрано {len(df1) - len(df2)}): {len(df2)} строк | окт/сент: {o2 / s2 - 1:.1%}")

Как считал менеджер: 5463 строк | окт/сент: -22.7%
Без дублей (убрано 180): 5283 строк | окт/сент: -19.6%
Без TEST (убрано 3): 5280 строк | окт/сент: -2.3%


Для менеджера: сильнее всего цифру исказили тестовые заказы TEST (всего 3 строки, но они искусственно раздули сентябрьскую выручку и сдвинули оценку падения с -19,6% до реальных -2,3%)

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [52]:
# 4
print("Пропусков в delivery_days:", clean_df["delivery_days"].isna().sum())
print("\nПо городам:\n", clean_df.loc[clean_df["delivery_days"].isna(), "city"].value_counts())
print("\nПо статусам:\n", clean_df.loc[clean_df["delivery_days"].isna(), "status"].value_counts())

d_sh = clean_df.loc[clean_df["city"] == "Шымкент", "delivery_days"]
print(f"\nШымкент <= 2 дня: (а) dropna = {(d_sh.dropna() <= 2).mean():.1%}, "
      f"(б) fillna(медиана={d_sh.median()}) = {(d_sh.fillna(d_sh.median()) <= 2).mean():.1%}, "
      f"(в) fillna(0) = {(d_sh.fillna(0) <= 2).mean():.1%}")

Пропусков в delivery_days: 158

По городам:
 city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64

По статусам:
 status
оплачен    144
отменён     14
Name: count, dtype: int64

Шымкент <= 2 дня: (а) dropna = 29.8%, (б) fillna(медиана=5.0) = 29.0%, (в) fillna(0) = 31.8%


Для менеджера: в чистых данных 158 пропусков в delivery_days (больше всего в Алматы 63 и Астане 40, по статусам 144 в оплачен и 14 в отменён). Честнее показывать dropna (29.8%), потому что fillna(медиана=5.0) (29.0%) подменяет неизвестные факты выдуманными значениями, а fillna(0) (31.8%) искусственно засчитывает пропуски как мгновенную доставку в 0 дней

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [89]:
# 5
# Ошибка 1: df.drop_duplicates() вызывается без inplace=True и без присваивания обратно в df, поэтому строки не удаляются
# Ошибка 2: .str.strip().str.lower().map(CITY) не применяется и остаются пробелы, латиница и кириллица в разных регистрах

CITY = {
    "алматы": "Алматы", "almaty": "Алматы",
    "астана": "Астана", "astana": "Астана",
    "шымкент": "Шымкент", "shymkent": "Шымкент",
    "караганда": "Караганда",
    "актобе": "Актобе",
    "атырау": "Атырау",
}

df_ai = pd.read_csv("orders.csv")
df_ai = df_ai.drop_duplicates().copy()
df_ai["city"] = df_ai["city"].str.strip().str.lower().map(CITY)
print(df_ai["city"].nunique(), len(df_ai))

6 5283


Для менеджера: после исправления обеих ошибок остаётся 6 городов и 5 283 строки без дублей, поэтому статистика по регионам больше не теряет заказы из-за пробелов, регистра и латиницы

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [71]:
# 6
dow = clean_df["order_date"].dt.day_name().value_counts()
print(dow)

min_day, max_day = dow.index[-1], dow.index[0]
min_val, max_val = dow.iloc[-1], dow.iloc[0]
diff_pct = (max_val - min_val) / max_val * 100
print(f"\nМинимум: {min_day} ({min_val}), максимум: {max_day} ({max_val}), меньше на {diff_pct:.1f}%")

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64

Минимум: Monday (684), максимум: Thursday (786), меньше на 13.0%


Для менеджера: меньше всего заказов в понедельник Monday = 684, а пик приходится на четверг Thursday = 786 в понедельник заказов на 13.0% меньше. График курьеров стоит делать гибким: меньше смен в понедельник и воскресенье и усиление на четверг–пятниц

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [83]:
# 7
sh_only = clean_df[clean_df["city"] == "Шымкент"]
print("Шымкент средний срок сентябрь:", round(sh_only.loc[sh_only["month"] == 9, "delivery_days"].mean(), 2),
      "| октябрь:", round(sh_only.loc[sh_only["month"] == 10, "delivery_days"].mean(), 2))

sh_valid = sh_only.dropna(subset=["delivery_days"]).sort_values("order_date")
sh_days = sh_valid["order_date"].dt.date
break_date = None
for dt in sorted(sh_days.unique()):
    if (sh_valid.loc[sh_days >= dt, "delivery_days"] >= 5).all():
        break_date = dt
        break
print("Первая дата, с которой срок в Шымкенте не меньше 5 дней:", break_date)

others_valid = clean_df[clean_df["city"] != "Шымкент"].dropna(subset=["delivery_days"])
others_days = others_valid["order_date"].dt.date
print("Другие города до даты:", round(others_valid.loc[others_days < break_date, "delivery_days"].mean(), 2),
      "| с даты:", round(others_valid.loc[others_days >= break_date, "delivery_days"].mean(), 2))

Шымкент средний срок сентябрь: 2.03 | октябрь: 6.54
Первая дата, с которой срок в Шымкенте не меньше 5 дней: 2026-10-01
Другие города до даты: 2.0 | с даты: 2.01


Для менеджера: в Шымкенте средний срок доставки вырос с 2.03 дня в сентябре до 6.54 дня в октябре, и начиная с 2026-10-01 заказы в Шымкенте не доставляются быстрее 5 дней. В первую очередь нужно проверить работу курьерской службы, склада и маршрутов в Шымкенте ровно с 1 октября

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [86]:
# 8
def to_amount(s):
    cleaned = s.str.replace(" ", "").str.replace(",", ".")
    return pd.to_numeric(cleaned, errors="coerce")

sample = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
print("Тест to_amount:\n", to_amount(sample))

amt_raw = pd.read_csv("orders.csv")["amount"]
amt_fn = to_amount(amt_raw)
amt_lesson = pd.to_numeric(amt_raw.str.replace(" ", ""))
print("Сумма to_amount:", amt_fn.sum(), "| Сумма с урока:", amt_lesson.sum(), "| Совпадает:", amt_fn.sum() == amt_lesson.sum())

Тест to_amount:
 0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64
Сумма to_amount: 109883360.0 | Сумма с урока: 109883360.0 | Совпадает: True


Для менеджера: функция корректно переводит строки с пробелами, точкой и запятой (15990.0, 15990.0, 1200.5), превращает мусор вроде "нет данных" в NaN без падения кода и на orders.csv даёт ту же сумму 109 883 360.0, что и расчёт на уроке